# Pasos previos

## Imports y dependencias

In [ ]:
pip install scikit-learn pandas numpy gensim nltk

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
import re
from nltk.tokenize import TweetTokenizer, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import GaussianNB, MultinomialNB
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.naive_bayes import GaussianNB


## Carga de archivo y preprocesamiento

In [ ]:
tknzr =  TweetTokenizer(preserve_case=False)
stemmer = SnowballStemmer('spanish')

nltk.download("stopwords", quiet=True)
stop_es = set(stopwords.words("spanish"))

def preprocesar(text): # Preprocesamiento hecho
  texto = text.strip()
  texto = tknzr.tokenize(texto)
  texto = [stemmer.stem(t) for t in texto if t not in stop_es]
  texto = ' '.join(texto)
  patron = r'[^\w\sáéíóúüñáàèìòùäëïöü]'
  texto = re.sub(patron, '', texto)
  texto = ' '.join(texto.split())
  return texto

In [ ]:
name = 'news_reducido.csv'

# Leer los datos en formato csv
data = pd.read_csv(name, on_bad_lines='skip')

# Nos quedamos con el texto (puedes quedarte con más información si quieres)
X = np.array([preprocesar(t) for t in data['text'].fillna('')])

## Datos utilizados

In [ ]:
semilla1=42
semilla2=640
semilla3=5300742

semilla = semilla2

#Agrupamiento
num_clusters=4

#Clasificación
pesos = ["uniform", "distance"]
valor_p = [1, 2]
valor_k = [3, 4, 5, 6, 7, 10]

#Clasificación
peso="uniform"
p=1
k=3


## Validación cruzada estratificada

In [ ]:
enc = OrdinalEncoder()
y = enc.fit_transform(np.reshape(data['category'], (-1,1))).reshape(-1) # DOCUMENTOS Y LE ASOCIO LA CATEGORIA

# Hacemos la partición train-test con Validacion cruzada estratificada
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf.get_n_splits(X, y)

# Agrupamiento

## Definición de los modelos

### Representación binaria

In [ ]:
text_kmeans_binario = Pipeline([
    ('vect', CountVectorizer(binary=True)),
    ('clf', KMeans(n_clusters=num_clusters, random_state=semilla))
])

### Representación de frecuencias

In [ ]:
text_kmeans_tf = Pipeline([
    ('vect', CountVectorizer(binary=False)),
    ('clf', KMeans(n_clusters=num_clusters, random_state=semilla))
])

### Representación TF-IDF

In [ ]:
text_kmeans_tfidf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', KMeans(n_clusters=num_clusters, random_state=semilla))
])

### Algoritmo de mezcla gaussiana

In [ ]:
text_kmeans_tfidf_mezcla_gausiana = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('to_dense', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
    ('clf', GaussianMixture(n_components=num_clusters, random_state=semilla, covariance_type='spherical'))
])

## Ejecución de los modelos

### Representación binaria

In [ ]:
text_kmeans = text_kmeans_binario

accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])
    # Test
    labels = text_kmeans.predict(X[tst])
    # Calculo de metricas
    acc = np.mean(labels == y[tst])
    print("Fold " + str(i+1) + ": " + str(acc))
    accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Representación de frecuencias

In [ ]:
text_kmeans = text_kmeans_tf

accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])
    # Test
    labels = text_kmeans.predict(X[tst])
    # Calculo de metricas
    acc = np.mean(labels == y[tst])
    print("Fold " + str(i+1) + ": " + str(acc))
    accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Representación TF-IDF

In [ ]:
text_kmeans = text_kmeans_tfidf

accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])
    # Test
    labels = text_kmeans.predict(X[tst])
    # Calculo de metricas
    acc = np.mean(labels == y[tst])
    print("Fold " + str(i+1) + ": " + str(acc))
    accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Algoritmo de mezcla gaussiana


In [ ]:
text_kmeans = text_kmeans_tfidf_mezcla_gausiana

accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])
    # Test
    labels = text_kmeans.predict(X[tst])
    # Calculo de metricas
    acc = np.mean(labels == y[tst])
    print("Fold " + str(i+1) + ": " + str(acc))
    accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interés, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

## Representación del mejor y el peor fold con el algoritmo T-SNE

### Peor fold: Representación: TF-IDF, Algoritmo: K-Means, Semilla: 42, Fold: 5

In [ ]:
# Este te saca las cosas con las fotos
text_kmeans = text_kmeans_tfidf

folds = list(skf.split(X,y))
tra, tst = folds[4]

# Entrenamiento
text_kmeans.fit(X[tra])
# Test
labels = text_kmeans.predict(X[tst])
# Calculo de metricas
acc = np.mean(labels == y[tst])
print(acc)

X_tst_transformed = text_kmeans[:-1].transform(X[tst])

tsne = TSNE(
    n_components=2,
    # Aseguramos que la perplejidad no sea mayor que el número de muestras
    perplexity=min(30, max(5, X_tst_transformed.shape[0] // 3)),
    learning_rate='auto',
    init='pca',
    random_state=42
)

tst_2d = tsne.fit_transform(X_tst_transformed)

plt.figure(figsize=(9, 6))
# Usamos 'labels' que son las predicciones que hiciste arriba
plt.scatter(tst_2d[:, 0], tst_2d[:, 1], c=labels, s=12, cmap='tab10', alpha=0.8)
plt.title(f"t-SNE fold {5} (Representación TF-IDF)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.colorbar(label="Cluster Predicho")
plt.tight_layout()
nombre_archivo = "./tsne_fold_5_clusters_tst.png"
plt.savefig(nombre_archivo, dpi=300, bbox_inches="tight")
print(f"Imagen guardada con éxito como: {nombre_archivo}")
plt.close()


# 1. Creamos un DataFrame con los nombres reales de las noticias del test
# y los IDs que el KMeans se ha inventado (labels)
df_mapping = pd.DataFrame({
    'KMeans_Cluster': labels,
    'Nombre_Real': data.loc[tst]['category'].values
})

# 2. Generamos la tabla de frecuencias
# Esto nos dirá cuántas noticias de cada categoría hay en cada clúster
tabla_frecuencias = pd.crosstab(df_mapping['KMeans_Cluster'], df_mapping['Nombre_Real'])

print("--- Tabla de Correspondencia ---")
print(tabla_frecuencias)

# 3. Código para decirte automáticamente cuál es cuál
print("\n--- Interpretación sugerida ---")
for cluster_id in tabla_frecuencias.index:
    # Buscamos la categoría con el valor máximo en esa fila
    categoria_predominante = tabla_frecuencias.columns[tabla_frecuencias.loc[cluster_id].argmax()]
    total_muestras = tabla_frecuencias.loc[cluster_id].sum()
    print(f"Clúster {cluster_id}: Probablemente es '{categoria_predominante}' ({total_muestras} noticias)")

### Mejor fold: Representación: TF-IDF, Algoritmo: K-Means, Semilla: 640, Fold: 2

In [ ]:
# Este te saca las cosas con las fotos
text_kmeans = text_kmeans_tfidf

folds = list(skf.split(X,y))
# Cambiado a la tercera posición (índice 2)
tra, tst = folds[1]

# Entrenamiento
text_kmeans.fit(X[tra])
# Test
labels = text_kmeans.predict(X[tst])
# Calculo de metricas
acc = np.mean(labels == y[tst])
print(acc)

print("Valores únicos en labels:", np.unique(labels))
print("Frecuencia por clúster:", np.bincount(labels))

X_tst_transformed = text_kmeans[:-1].transform(X[tst])

tsne = TSNE(
    n_components=2,
    # Aseguramos que la perplejidad no sea mayor que el número de muestras
    perplexity=min(30, max(5, X_tst_transformed.shape[0] // 3)),
    learning_rate='auto',
    init='pca',
    random_state=42
)

tst_2d = tsne.fit_transform(X_tst_transformed)

plt.figure(figsize=(9, 6))
# Usamos 'labels' que son las predicciones que hiciste arriba
plt.scatter(tst_2d[:, 0], tst_2d[:, 1], c=labels, s=12, cmap='tab10', alpha=0.8)
# Actualizado el título y el nombre del archivo al Fold 3
plt.title(f"t-SNE fold {2} (Representación TF-IDF)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.colorbar(label="Cluster Predicho")
plt.tight_layout()
nombre_archivo = "./tsne_fold_2_clusters_tst.png"
plt.savefig(nombre_archivo, dpi=300, bbox_inches="tight")
print(f"Imagen guardada con éxito como: {nombre_archivo}")
plt.close()


# 1. Creamos un DataFrame con los nombres reales de las noticias del test
# y los IDs que el KMeans se ha inventado (labels)
df_mapping = pd.DataFrame({
    'KMeans_Cluster': labels,
    'Nombre_Real': data.loc[tst]['category'].values
})

# 2. Generamos la tabla de frecuencias
# Esto nos dirá cuántas noticias de cada categoría hay en cada clúster
tabla_frecuencias = pd.crosstab(df_mapping['KMeans_Cluster'], df_mapping['Nombre_Real'])

print("--- Tabla de Correspondencia ---")
print(tabla_frecuencias)

# 3. Código para decirte automáticamente cuál es cuál
print("\n--- Interpretación sugerida ---")
for cluster_id in tabla_frecuencias.index:
    # Buscamos la categoría con el valor máximo en esa fila
    categoria_predominante = tabla_frecuencias.columns[tabla_frecuencias.loc[cluster_id].argmax()]
    total_muestras = tabla_frecuencias.loc[cluster_id].sum()
    print(f"Clúster {cluster_id}: Probablemente es '{categoria_predominante}' ({total_muestras} noticias)")

### Mejor combinación binaria: Algoritmo: K-Means, Semilla: 640, Fold: 1

In [ ]:
# Este te saca las cosas con las fotos
text_kmeans = text_kmeans_binario

folds = list(skf.split(X,y))
tra, tst = folds[0]

# Entrenamiento
text_kmeans.fit(X[tra])
# Test
labels = text_kmeans.predict(X[tst])
# Calculo de metricas
acc = np.mean(labels == y[tst])
print(acc)

X_tst_transformed = text_kmeans[:-1].transform(X[tst])

tsne = TSNE(
    n_components=2,
    # Aseguramos que la perplejidad no sea mayor que el número de muestras
    perplexity=min(30, max(5, X_tst_transformed.shape[0] // 3)),
    learning_rate='auto',
    init='pca',
    random_state=42
)

tst_2d = tsne.fit_transform(X_tst_transformed)

plt.figure(figsize=(9, 6))
# Usamos 'labels' que son las predicciones que hiciste arriba
plt.scatter(tst_2d[:, 0], tst_2d[:, 1], c=labels, s=12, cmap='tab10', alpha=0.8)
plt.title(f"t-SNE fold {1} (Representación Binaria)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.colorbar(label="Cluster Predicho")
plt.tight_layout()
nombre_archivo = "./tsne_fold_1_clusters_tst.png"
plt.savefig(nombre_archivo, dpi=300, bbox_inches="tight")
print(f"Imagen guardada con éxito como: {nombre_archivo}")
plt.close()

### Mejor combinación frecuencias: Algoritmo: K-Means, Semilla: 640, Fold: 2

In [ ]:
# Este te saca las cosas con las fotos
text_kmeans = text_kmeans_tf

folds = list(skf.split(X, y))
tra, tst = folds[1]

# Entrenamiento
text_kmeans.fit(X[tra])
# Test
labels = text_kmeans.predict(X[tst])
# Calculo de metricas
acc = np.mean(labels == y[tst])
print(acc)

X_tst_transformed = text_kmeans[:-1].transform(X[tst])

tsne = TSNE(
    n_components=2,
    # Aseguramos que la perplejidad no sea mayor que el número de muestras
    perplexity=min(30, max(5, X_tst_transformed.shape[0] // 3)),
    learning_rate='auto',
    init='pca',
    random_state=42
)

tst_2d = tsne.fit_transform(X_tst_transformed)

plt.figure(figsize=(9, 6))
# Usamos 'labels' que son las predicciones que hiciste arriba
plt.scatter(tst_2d[:, 0], tst_2d[:, 1], c=labels, s=12, cmap='tab10', alpha=0.8)
plt.title(f"t-SNE fold {2} (Representación frecuencias)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.colorbar(label="Cluster Predicho")
plt.tight_layout()
nombre_archivo = "./tsne_fold_2_clusters_tst.png"
plt.savefig(nombre_archivo, dpi=300, bbox_inches="tight")
print(f"Imagen guardada con éxito como: {nombre_archivo}")
plt.close()

# Clasificación

## Definición de los modelos

### Binaria

In [ ]:
text_clasifier_binario = Pipeline([
    ('vect', CountVectorizer(binary=True)),
    ('clf', KNeighborsClassifier()) #Preguntar los randoms state
])

### Frecuencias

In [ ]:
text_clasifier_tf = Pipeline([
    ('vect', CountVectorizer(binary=False)),
    ('clf', KNeighborsClassifier()) #Preguntar el random_state
])

### TF-IDF

In [ ]:
text_clasifier_tfidf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', KNeighborsClassifier())
])

### Gausiana Binaria

In [ ]:
text_Gausian_binaria = Pipeline([
    ('vect', CountVectorizer(binary=True)),
    ('to_dense', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
    ('clf', GaussianNB())
])

### Gausiana Frecuencias

In [ ]:
text_Gausian_frecuencias = Pipeline([
    ('vect', CountVectorizer(binary=False)),
    ('to_dense', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
    ('clf', GaussianNB())
])

### Gausiana TF-IDF


In [ ]:
text_Gausian_tfidf = Pipeline([ # Solo para TF-IDF # No funciona en el servidor, se caga encima
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('to_dense', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
    ('clf', GaussianNB())
])

### Multinomial binario

In [ ]:
text_Multinomial_binario = Pipeline([
    ('vect', CountVectorizer(binary=True)),
    ('clf', MultinomialNB())
])

### Multinomial frecuencias

In [ ]:
text_Multinomial_frecuencias = Pipeline([
    ('vect', CountVectorizer(binary=False)),
    ('clf', MultinomialNB())
])

### Multinomial TF-IDF

In [ ]:
text_Multinomial_tfidf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB())
])

## Ejecución de los modelos

### Binaria

In [ ]:
text_sgd = text_clasifier_binario

text_sgd.set_params(
    clf__weights=peso,
    clf__p=p,
    clf__n_neighbors=k
)

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])

    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Frecuencias

In [ ]:
text_sgd = text_clasifier_tf

text_sgd.set_params(
    clf__weights=peso,
    clf__p=p,
    clf__n_neighbors=k
)

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])

    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### TF-IDF

In [ ]:
text_sgd = text_clasifier_tfidf

text_sgd.set_params(
    clf__weights=peso,
    clf__p=p,
    clf__n_neighbors=k
)

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])

    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Gausiana binaria

In [ ]:
text_sgd = text_Gausian_binaria

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])

    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Gausiana frecuencias

In [ ]:
text_sgd = text_Gausian_frecuencias

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])

    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Gausiana TF-IDF

In [ ]:
text_sgd = text_Gausian_tfidf

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])

    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Multinomial binario

In [ ]:
text_sgd = text_Multinomial_binario

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])

    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Multinomial frecuencias

In [ ]:
text_sgd = text_Multinomial_frecuencias

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])

    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Multinomial TF-IDF

In [ ]:
text_sgd = text_Multinomial_tfidf

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])
    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Script para la automatización de las pruebas de los agoritmos de K-NN

In [ ]:
pesos = ["uniform", "distance"]
valor_p = [1, 2]
valor_k = [3, 4, 5, 6, 7, 10]

results = []

pipelines = {
    'binary': text_clasifier_binario,
    'tf':     text_clasifier_tf,
    'tfidf':  text_clasifier_tfidf
}


for nombre_pipe, pipeline in pipelines.items():
    for peso in pesos:
        for p in valor_p:
            for k in valor_k:
                # Actualizar hiperparámetros del clasificador dentro del pipeline
                pipeline.set_params(
                    clf__weights=peso,
                    clf__p=p,
                    clf__n_neighbors=k
                )

                accuracies = np.zeros(5)
                for i, (tra, tst) in enumerate(skf.split(X, y)):
                    pipeline.fit(X[tra], y[tra])
                    predicted = pipeline.predict(X[tst])
                    accuracies[i] = np.mean(predicted == y[tst])
                    print("Fold " + str(i+1) +": " + str(accuracies[i]))

                avg_acc = np.average(accuracies)
                results.append({
                    'semilla':     semilla,
                    'vectorizer':  nombre_pipe,
                    'peso':        peso,
                    'p':           p,
                    'k':           k,
                    'avg_acc':     avg_acc
                })
                print(f'semilla={semilla} | vec={nombre_pipe} | peso={peso} | p={p} | k={k} → acc={avg_acc}')


In [ ]:
# Convertir a DataFrame para analizar mejor los resultados
results_df = pd.DataFrame(results)
top_3 = results_df.nlargest(3, 'avg_acc')

bottom_2 = results_df.nsmallest(2, 'avg_acc')

print("=== TOP 3 MEJORES CONFIGURACIONES ===")
print(top_3)

print("\n=== 2 PEORES CONFIGURACIONES ===")
print(bottom_2)